In [ ]:
from ConvAE import Conv_AE, Conv_AE_diff_targ
from autoEncoder import AutoEncoder_desp
import copy
import torch.nn.functional as F
import torchvision.transforms.functional as F_trans
import torch
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader, Subset
from torchvision import datasets
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
import albumentations as A
import random
import imutils
import pandas as pd
import math, matplotlib.pyplot as plt, numpy as np
import os
import json
import pickle
from sklearn.model_selection import KFold

# Utilitys

Run all functions before continouing.

This section defines helper functions used for inference, distance calculations, data creation, plotting, bootstrapping, ISA, and augmentation. 

In [ ]:
# ====================== Inference on ISA =================

def ratio_inference(init_R, conv_R):
    """Find overlap ratios between initial and converged row sets.
    
    Input:
    init_R [list]: Initial row-index groups used to start ISA.
    conv_R [list]: Converged row-index arrays aligned with init_R.
    
    Output:
    ratio_cor_act_and_CONV_R_dict [dict]: Overlap proportions for each converged row set.
    length_of_clump [dict]: Number of samples in each converged row set.
    
    """
    ratio_cor_act_and_CONV_R_dict = {"Ratio convCAPinit-act to conv-act": []}
    length_of_clump = {"Converged sample length": []}
    
    n = len(init_R)
    
    for i in range(n):
        if len(conv_R[i]) > 0:
            union_correct_new = np.intersect1d(init_R[i][0], conv_R[i]) 
                    
            ratio_cor_act_and_conv_R = len(union_correct_new)/(conv_R[i].shape[0])

            ratio_cor_act_and_CONV_R_dict["Ratio convCAPinit-act to conv-act"].append(ratio_cor_act_and_conv_R)
            length_of_clump["Converged sample length"].append(conv_R[i].shape[0])
        else:
            length_of_clump["Converged sample length"].append(np.nan)
            ratio_cor_act_and_CONV_R_dict["Ratio convCAPinit-act to conv-act"].append(np.nan)

    return ratio_cor_act_and_CONV_R_dict, length_of_clump 



def find_act_given_initialization(data, list_indecis, tau, num_itr):
    """Run ISA from provided row initializations.
    
    Input:
    data [array-like]: Matrix used for ISA row and column updates.
    list_indecis [list]: Initial row-index groups.
    tau [float]: Threshold multiplier for standard deviation.
    num_itr [int]: Maximum number of ISA iterations.
    
    Output:
    conv_R_list [list]: Converged row-index sets for each initialization.
    cnt [int]: Count of empty or non-converged clumps."""

    X = np.array(data)
    
    conv_R_list = []
    
    cnt = 0
    
    for init_rows in list_indecis:

        XR = standardize_rows(X)
        XC = standardize_cols(X)


        
        R = np.array(init_rows[0])

        C = np.array([], dtype=int)
        

        for k in range(num_itr):

            prev_R = R.copy()
            prev_C = C.copy()

            # Step 2: Find columns where sum is tau std above mean
            if len(R) > 0:
                col_sums = XC[R, :].sum(axis=0) 
                mean_sum = col_sums.mean()
                std_sum = col_sums.std()
                if std_sum > 0:
                    threshold = mean_sum + tau * std_sum
                    C = np.where(col_sums > threshold)[0]

            # Step 3: Find rows where sum is tau std above mean
            if len(C) > 0:
                row_sums = XR[:, C].sum(axis=1)
                mean_sum = row_sums.mean()
                std_sum = row_sums.std()
                if std_sum > 0:
                    threshold = mean_sum + tau * std_sum
                    R = np.where(row_sums > threshold)[0]
                    
            # Check convergence
            if len(R) == len(prev_R) and len(C) == len(prev_C):
                if len(R) > 0 and len(C) > 0:
                    if np.array_equal(np.sort(R), np.sort(prev_R)) and \
                    np.array_equal(np.sort(C), np.sort(prev_C)):
                        #print(" Convergence ")
                        break
                    
            # Empty clump check
            if len(R) == 0 or len(C) == 0:
                R = []
                cnt += 1
                break
            
            if k == (num_itr - 1):
                print("\nmax iterations meat\n")
        
        conv_R_list.append(R)
            
        
    return conv_R_list, cnt

def random_indecis_list(data, number_init, row_size, seed=None):
    """Sample random row-index groups for ISA initialization.
    
    Input:
    data [array-like]: Matrix whose rows are sampled.
    number_init [int]: Number of initialization groups to create.
    row_size [int]: Number of row indices per group.
    seed [int | None]: Random seed for reproducible sampling.
    
    Output:
    index_list [list]: List of sampled row-index tuples."""
    rng = np.random.default_rng(seed)

    index_list = []
    for _ in range(number_init):
        temp = (rng.choice(data.shape[0], size=row_size, replace=False),)
        index_list.append(temp)

    return index_list


def mean_std(data, number_init, row_size, tau, num_itr, precentage_must_converge=0.2, index_list=None, seed = None):
    """Summarize random-initialization overlap ratios.
    
    Input:
    data [array-like]: Matrix used for ISA.
    number_init [int]: Number of random initialization groups.
    row_size [int]: Number of row indices per initialization.
    tau [float]: Threshold multiplier for ISA updates.
    num_itr [int]: Maximum number of ISA iterations.
    precentage_must_converge [float]: Minimum converged fraction required for valid statistics.
    index_list [list | None]: Optional fixed initialization groups.
    seed [int | None]: Random seed for generated initializations.
    
    Output:
    mean_init [float]: Mean overlap with initial row sets.
    mean_conv [float]: Mean overlap with converged row sets.
    std_init [float]: Standard deviation of initial overlaps.
    std_conv [float]: Standard deviation of converged overlaps."""
    if index_list is None:

        index_list = random_indecis_list(data, number_init, row_size, seed = seed)
    
    activations_data, cnt = find_act_given_initialization(data, 
                                index_list, 
                                tau,
                                num_itr=num_itr)
    
    lower_bound = np.ceil(len(activations_data)*precentage_must_converge)
    if len(activations_data) - cnt > lower_bound:
        
        # Inference

        dict_init, dict_conv = ratio_inference(index_list, activations_data)

    
    
    
        # convert from dictonary to pandas dataframe
        df_dict_ratio = pd.DataFrame({
            'State': range(len(dict_init["Ratio convCAPinit-act to init-act"])),
            "corr act / init": dict_init["Ratio convCAPinit-act to init-act"],
            "corr act / new": dict_conv["Ratio convCAPinit-act to conv-act"]
        })
        
        # calcualte mean of dict_init and dict_conv
        mean_init = df_dict_ratio["corr act / init"].mean() 
        mean_conv = df_dict_ratio["corr act / new"].mean()

        # calcualte std of dict_init and dict_conv
        std_init = df_dict_ratio["corr act / init"].std()
        std_conv = df_dict_ratio["corr act / new"].std()
        
        return mean_init, mean_conv, std_init, std_conv

    else:
        print("Less then 50 initializations converged to valid value, number of not converged: ", cnt)
        return np.nan, np.nan, np.nan, np.nan


def compute_ratios(data, number_init, row_size,  tau, num_itr=100, seed=None):
    """Compute ISA overlap ratios for random initializations.
    
    Input:
    data [array-like]: Matrix used for ISA.
    number_init [int]: Number of initialization groups.
    row_size [int]: Number of row indices per initialization.
    tau [float]: Threshold multiplier for ISA updates.
    num_itr [int]: Maximum number of ISA iterations.
    seed [int | None]: Random seed for initialization sampling.
    
    Output:
    df_dict_ratio [pd.DataFrame]: Proportion and converged-sample counts."""

    index_list = random_indecis_list(data, number_init, row_size, seed = seed)

    activations_data, cnt = find_act_given_initialization(data, 
                                                          list_indecis=index_list, 
                                                          tau=tau, 
                                                          num_itr=num_itr)
    


    dict_conv, dict_size = ratio_inference(index_list, activations_data)

    
    
    
    df_dict_ratio = pd.DataFrame({
        "Proportion": dict_conv["Ratio convCAPinit-act to conv-act"],
        "Number of samples": dict_size["Converged sample length"]
    })
        
    return df_dict_ratio


def compare_random_init_corr_init(data, number_init, row_size, tau, num_itr, std_threashold,print_stat=False, precentage_must_converge = 0.2, index_list = None, seed = None):
    """Compare random and correct ISA initializations.
    
    Input:
    data [array-like]: Matrix used for ISA.
    number_init [int]: Number of initialization groups.
    row_size [int]: Number of row indices per initialization.
    tau [float]: Threshold multiplier for ISA updates.
    num_itr [int]: Maximum number of ISA iterations.
    std_threashold [float]: Standard-deviation threshold for comparison prints.
    print_stat [bool]: Whether to print comparison messages.
    precentage_must_converge [float]: Minimum converged fraction required for statistics.
    index_list [list | None]: Optional correct initialization groups.
    seed [int | None]: Random seed for random initializations.
    
    Output:
    random_data [tuple]: Mean and standard-deviation statistics for random initializations.
    correct_data [tuple]: Mean and standard-deviation statistics for provided initializations."""
    mean_init_rand, mean_conv_rand, std_init_rand, std_conv_rand = mean_std(data,
                                                                            number_init=number_init,
                                                                            range_init=row_size,
                                                                            tau=tau,
                                                                            precentage_must_converge=precentage_must_converge,
                                                                            num_itr=num_itr,
                                                                            index_list=None,
                                                                            seed = seed)
    
    random_data = (mean_init_rand, mean_conv_rand, std_init_rand, std_conv_rand)
    correct_data = (np.nan, np.nan, np.nan, np.nan)
    if index_list != None:
        mean_init_cor, mean_conv_cor, std_init_cor, std_conv_cor = mean_std(data,
                                                                            number_init=number_init,
                                                                            range_init=row_size,
                                                                            tau=tau,
                                                                            num_itr=num_itr,
                                                                            index_list=index_list,
                                                                            seed = seed)
        
    
        correct_data = (mean_init_cor, mean_conv_cor, std_init_cor, std_conv_cor)
    if index_list != None and print_stat ==True:
        if mean_init_rand - std_threashold*std_init_rand > mean_init_cor or mean_init_rand + std_threashold*std_init_rand < mean_init_cor:
            print("\nMean of ration intersect / correct initialization is ", std_threashold, " std away from random initialization.\n")
        else:
            print("\nNothing indecating significant 1 \n")
        
    
        if mean_conv_rand - std_threashold*std_conv_rand > mean_conv_cor or mean_conv_rand + std_threashold*std_conv_rand < mean_conv_cor:
            print("\nMean of ration intersect / convergence row activation is ", std_threashold, " std away from row convergence of random initialization.\n")
        else:
            print("\nNothing indecating significant 2 \n")
        
    return random_data, correct_data


In [ ]:
# ========================== Creat data ===========================

def rotate_images(images, angle):
    """Rotate each image in a batch.
    
    Input:
    images [np.ndarray]: Batch of images to rotate.
    angle [float]: Rotation angle in degrees.
    
    Output:
    rotated_images [np.ndarray]: Batch of rotated images."""
    
    
    rotated_images = []
    for img in images:
        rotated = imutils.rotate(img, angle=angle) 
        rotated_images.append(rotated)
    return np.array(rotated_images)

def pad_shift_images(imgs, rng=None, square=False, pad=50):
    """Pad images and apply random spatial shifts.
    
    Input:
    imgs [np.ndarray]: Batch of images to pad and shift.
    rng [np.random.Generator | int | None]: Random generator or seed.
    square [bool]: Whether to use doubled horizontal padding.
    pad [int]: Padding width and shift range.
    
    Output:
    shifted_imgs [np.ndarray]: Batch of padded and shifted images."""
    
    square_adj = 1
    
    if square:
        square_adj=2
    
    tensor_imgs = torch.from_numpy(imgs)
    tensor_imgs_pad = F.pad(tensor_imgs, (square_adj*pad, square_adj*pad, pad, pad), mode='constant', value=0)
    
    size = tensor_imgs_pad.shape[0]
    
    if rng is None:
        rng = np.random.default_rng()
    elif isinstance(rng, int):
        rng = np.random.default_rng(rng)
        
    shifts = rng.uniform(-pad, pad, (size, 2)).tolist()
    
    shifted_imgs = [
        F_trans.affine(img.unsqueeze(0), angle=0, translate=shift, scale=1.0, shear=(0, 0))
        .squeeze(0)
        for img, shift in zip(tensor_imgs_pad, shifts)
    ]
    
    return np.array(shifted_imgs)

# %%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%%

def creat_rotated_data_incl_no_pad(data_a, data_b, sigma_list, theta_list, seed, pad= 50): 
    """Create rotated paired datasets with padded and non-padded targets.
    
    Input:
    data_a [np.ndarray]: First image batch.
    data_b [np.ndarray]: Second image batch.
    sigma_list [list]: Rotation angles for data_a.
    theta_list [list]: Rotation angles for data_b.
    seed [int]: Random seed for shuffling and shifts.
    pad [int]: Padding width for shifted targets.
    
    Output:
    combined_data_normalized [np.ndarray]: Normalized non-padded paired data.
    combined_data_normalized_pad [np.ndarray]: Normalized padded paired data.
    all_labels_shuffled [np.ndarray]: Shuffled combined labels."""
    # Rotate the 3s and 7s by different angles
    
    rng = np.random.default_rng(seed)
    
    data_a_rotated_list = []
    data_b_rotated_list = []
    data_a_rotated_list_pad = []
    data_b_rotated_list_pad = []
    
    labels_a_rotated_list = []
    labels_b_rotated_list = []
    
    k = 0
    for sigma in sigma_list:
        # rotate images
        data_temp = rotate_images(data_a, sigma)
        data_tep_pad = pad_shift_images(imgs=data_temp, rng=rng, pad=pad, square=False)
        
        # creat labels
        labels_temp = np.ones(len(data_temp))*k
        
        # append rotated data to list
        data_a_rotated_list.append(data_temp)
        data_a_rotated_list_pad.append(data_tep_pad)
        
        # append labels to label list
        labels_a_rotated_list.append(labels_temp)
        k += 1
    
    
    k = 0 
    for theta in theta_list:
        data_temp = rotate_images(data_b, theta)
        data_tep_pad = pad_shift_images(imgs=data_temp, rng=rng, pad=pad, square=False)
        
        labels_temp = np.ones(len(data_temp))*k
        
        data_b_rotated_list.append(data_temp)
        data_b_rotated_list_pad.append(data_tep_pad)
        
        labels_b_rotated_list.append(labels_temp)
        k += 10

    # Combine all rotated data with original data

    all_data_a = np.zeros((1, data_a_rotated_list[0][0].shape[0], data_a_rotated_list[0][0].shape[1]))
    all_data_b = np.zeros((1, data_b_rotated_list[0][0].shape[0], data_b_rotated_list[0][0].shape[1]))
    all_data_a_pad = np.zeros((1, data_a_rotated_list_pad[0][0].shape[0], data_a_rotated_list_pad[0][0].shape[1]))
    all_data_b_pad = np.zeros((1, data_b_rotated_list_pad[0][0].shape[0], data_b_rotated_list_pad[0][0].shape[1]))
    
    all_labels_a = np.zeros(1)
    all_labels_b = np.zeros(1)
    
    itr = 0
    for data_a_rotated in data_a_rotated_list:
        all_data_a = np.concatenate([all_data_a, data_a_rotated])
        all_labels_a = np.concatenate([all_labels_a, labels_a_rotated_list[itr]])
        
        itr += 1
    
    itr = 0
    for data_b_rotated in data_b_rotated_list:
        all_data_b = np.concatenate([all_data_b, data_b_rotated])
        all_labels_b = np.concatenate([all_labels_b, labels_b_rotated_list[itr]])
        
        itr += 1
    
    itr = 0
    for data_a_rotated_pad in data_a_rotated_list_pad:
        all_data_a_pad = np.concatenate([all_data_a_pad, data_a_rotated_pad])
        
        itr += 1
    
    itr = 0
    for data_b_rotated_pad in data_b_rotated_list_pad:
        all_data_b_pad = np.concatenate([all_data_b_pad, data_b_rotated_pad])
        
        itr += 1
        
    all_data_a = all_data_a[1:] 
    all_data_b = all_data_b[1:]
    all_data_a_pad = all_data_a_pad[1:]
    all_data_b_pad = all_data_b_pad[1:]
    
    all_labels_a = all_labels_a[1:]
    all_labels_b = all_labels_b[1:]

    
    if len(all_data_a) < len(all_data_b):
        shuffle_indices_b = rng.permutation(len(all_data_b))
        shuffle_indices_b = shuffle_indices_b[:len(all_data_a)]
        shuffle_indices_a = rng.permutation(len(all_data_a))
        
    elif len(all_data_a) > len(all_data_b):
        shuffle_indices_a = rng.permutation(len(all_data_a))
        shuffle_indices_a = shuffle_indices_a[:len(all_data_b)]
        shuffle_indices_b = rng.permutation(len(all_data_b))
        
    else:
        shuffle_indices_a = rng.permutation(len(all_data_a))
        shuffle_indices_b = rng.permutation(len(all_data_b))
    
    # Shuffle the data and labels
    all_data_a_shuffled = all_data_a[shuffle_indices_a]
    all_data_b_shuffled = all_data_b[shuffle_indices_b]
    all_data_a_shuffled_pad = all_data_a_pad[shuffle_indices_a]
    all_data_b_shuffled_pad = all_data_b_pad[shuffle_indices_b]

    all_labels_a_shuffled = all_labels_a[shuffle_indices_a]
    all_labels_b_shuffled = all_labels_b[shuffle_indices_b]



    # Fix label array so it has values 0, 1, 2,..., n 
    all_labels_shuffled = all_labels_a_shuffled + all_labels_b_shuffled
    all_labels_shuffled_uniqe = list(np.unique(all_labels_shuffled))

    itr = 0
    for i in all_labels_shuffled_uniqe:
        all_labels_shuffled[all_labels_shuffled == i] = itr
        itr += 1


    # flatten the data from (num_samples, 28, 28) to (num_samples, 784)
    all_data_a_shuffled_flattened = all_data_a_shuffled.reshape(all_data_a_shuffled.shape[0], -1)
    all_data_b_shuffled_flattened = all_data_b_shuffled.reshape(all_data_b_shuffled.shape[0], -1)
    all_data_a_shuffled_flattened_pad = all_data_a_shuffled_pad.reshape(all_data_a_shuffled_pad.shape[0], -1)
    all_data_b_shuffled_flattened_pad = all_data_b_shuffled_pad.reshape(all_data_b_shuffled_pad.shape[0], -1)


    # concatinate the data and get the shape (num_samples, 1568)
    combined_data = np.concatenate([all_data_a_shuffled_flattened, all_data_b_shuffled_flattened], axis=1)
    combined_data_pad = np.concatenate([all_data_a_shuffled_flattened_pad, all_data_b_shuffled_flattened_pad], axis=1)


    # normalize the data
    combined_data_normalized = combined_data.astype(np.float32) / np.max(combined_data)
    combined_data_normalized_pad = combined_data_pad.astype(np.float32) / np.max(combined_data_pad)
    
    return combined_data_normalized, combined_data_normalized_pad, all_labels_shuffled



In [ ]:
# =================== Plot functions

def plot_augmented_comparison(idx, data_aug_shuffle, data_aug_targ_shuffle, data_aug_targ_Npad_shuffle, cmap='gray', figsize=(15, 5)):
    """Plot one augmented sample beside its targets.
    
    Input:
    idx [int]: Sample index to plot.
    data_aug_shuffle [np.ndarray]: Augmented input images.
    data_aug_targ_shuffle [np.ndarray]: Padded target images.
    data_aug_targ_Npad_shuffle [np.ndarray]: Non-padded target images.
    cmap [str]: Matplotlib colormap.
    figsize [tuple]: Figure size.
    
    Output:
    None [NoneType]: Displays the figure."""
    import matplotlib.pyplot as plt
    fig, axes = plt.subplots(1, 3, figsize=figsize)
    
    axes[0].imshow(data_aug_shuffle[idx], cmap=cmap)
    axes[0].set_title('Augmented')
    axes[0].axis('off')
    
    axes[1].imshow(data_aug_targ_shuffle[idx], cmap=cmap)
    axes[1].set_title('Target Padded')
    axes[1].axis('off')
    
    axes[2].imshow(data_aug_targ_Npad_shuffle[idx], cmap=cmap)
    axes[2].set_title('Target No Pad')
    axes[2].axis('off')
    
    plt.tight_layout()
    plt.show()
    

def plot_org_only(indices, data_org, cmap='gray', figsize_per_row=(6,3)):
    """Plot original paired samples side by side.
    
    Input:
    indices [list]: Sample indices to plot.
    data_org [np.ndarray]: Original paired samples.
    cmap [str]: Matplotlib colormap.
    figsize_per_row [tuple]: Figure size per plotted row.
    
    Output:
    None [NoneType]: Displays the figure."""
    # support (N, H, W) or (N, D) where D = 2 * side^2
    arr = np.asarray(data_org)
    if arr.ndim == 3:
        arr = arr.reshape(arr.shape[0], -1)
    rows = len(indices)
    fig, axes = plt.subplots(rows, 1, figsize=(figsize_per_row[0], figsize_per_row[1]*rows))
    
    # Ensure axes is always indexable as a 1D array
    if rows == 1:
        axes = [axes]
    
    for i, idx in enumerate(indices):
        org = arr[idx]
        half = org.size // 2
        left = org[:half]; right = org[half:]
        side = int(math.sqrt(left.size))
        assert side * side == left.size, "left half is not square"
        img = np.hstack([left.reshape(side, side), right.reshape(side, side)])
        ax = axes[i]
        ax.imshow(img, cmap=cmap)
        ax.axis('off')
        ax.set_title(f'org idx {idx}')
    plt.tight_layout()

In [ ]:
# ======================= Bootstrap functions ===================

def bootstrap_mean_ci(
    data: np.ndarray,
    n_bootstrap: int = 1000,
    alpha: float = 0.05,
    bins: int = 30,
    random_state: int | None = None,
):
    """Estimate a mean confidence interval by bootstrap resampling.
    
    Input:
    data [np.ndarray]: One-dimensional data array.
    n_bootstrap [int]: Number of bootstrap samples.
    alpha [float]: Significance level for the confidence interval.
    bins [int]: Histogram bin count if plotting is enabled.
    random_state [int | None]: Random seed for bootstrap sampling.
    
    Output:
    results [dict]: Sample mean, bootstrap means, CI bounds, and metadata."""
    data = np.asarray(data)

    if data.ndim != 1:
        raise ValueError("data must be a 1D NumPy array")
    if len(data) == 0:
        raise ValueError("data must not be empty")
    if n_bootstrap <= 0:
        raise ValueError("n_bootstrap must be > 0")
    if not (0 < alpha < 1):
        raise ValueError("alpha must be between 0 and 1")

    rng = np.random.default_rng(random_state)
    n = len(data)

    bootstrap_means = np.empty(n_bootstrap, dtype=float)

    for i in range(n_bootstrap):
        sample = data[rng.integers(0, n, size=n)]
        bootstrap_means[i] = np.mean(sample)

    sample_mean = np.mean(data)
    lower_bound = np.percentile(bootstrap_means, 100 * (alpha / 2))
    upper_bound = np.percentile(bootstrap_means, 100 * (1 - alpha / 2))
    bootstrap_std = np.std(bootstrap_means, ddof=1)

    return {
        "sample_mean": sample_mean,
        "bootstrap_means": bootstrap_means,
        "bootstrap_std": bootstrap_std,
        "lower_bound": lower_bound,
        "upper_bound": upper_bound,
        "alpha": alpha,
        "n_bootstrap": n_bootstrap,
        "n_samples": n,
        "random_state": random_state,
    }
    

def run_ratio_bootstrap_analysis(
    data,
    T,
    number_init: int,
    num_itr: int,
    alpha: float,
    n_bootstrap: int = 10000,
    range_init = [755, 790]
):
    """Run bootstrap summaries across tau values.
    
    Input:
    data [array-like]: Matrix used for ISA ratio computation.
    T [list]: Tau values to evaluate.
    number_init [int]: Number of random initialization groups.
    num_itr [int]: Maximum number of ISA iterations.
    alpha [float]: Significance level for bootstrap intervals.
    n_bootstrap [int]: Number of bootstrap samples.
    range_init [list]: Row-index range or size passed to ratio computation.
    
    Output:
    results [list]: Bootstrap summaries for each tau value."""
    results = []

    seed_ratios = 1001
    seed_bootstrap_1 = 2001
    seed_bootstrap_2 = 3001

    for tau in T:
        pd_ratio = compute_ratios(
            data=data,
            number_init=number_init,
            range_init=range_init,
            tau=tau,
            num_itr=num_itr,
            seed=seed_ratios,
        )
        

        #else:
        Proportion_list = pd_ratio["Proportion"].dropna()
        size_converged_set_list = pd_ratio["Number of samples"].dropna()
        

        if len(Proportion_list) == 0:
            print("\nRatio 1 list empty\n")
            ratio_1_stat={
                "sample_mean": np.nan,
                "bootstrap_means": np.nan,
                "bootstrap_std": np.nan,
                "lower_bound": np.nan,
                "upper_bound": np.nan,
                "alpha": np.nan,
                "n_bootstrap": np.nan,
                "n_samples": np.nan,
                "random_state": np.nan,
            }
        else:
            ratio_1_stat = bootstrap_mean_ci(
                data=Proportion_list,
                n_bootstrap=n_bootstrap,
                alpha=alpha,
                random_state=seed_bootstrap_1,
            )
        
        if len(size_converged_set_list) == 0:
            print("\nsize_converged_set list empty\n")
            converges_set_size_stat={
                "sample_mean": np.nan,
                "bootstrap_means": np.nan,
                "bootstrap_std": np.nan,
                "lower_bound": np.nan,
                "upper_bound": np.nan,
                "alpha": np.nan,
                "n_bootstrap": np.nan,
                "n_samples": np.nan,
                "random_state": np.nan,
            }
        else:
            converges_set_size_stat = bootstrap_mean_ci(
                data=size_converged_set_list,
                n_bootstrap=n_bootstrap,
                alpha=alpha,
                random_state=seed_bootstrap_2,
            )
        

        results.append({
            "tau": tau,
            "pd_ratio_small": pd_ratio,
            "range_init": range_init,
            "converges_set_size_stat": converges_set_size_stat,
            "ratio_1_BS_stat": ratio_1_stat,
            "seed_ratios": seed_ratios,
            "seed_bootstrap_1": seed_bootstrap_1,
            "seed_bootstrap_2": seed_bootstrap_2,
        })

        seed_ratios += 1
        seed_bootstrap_1 += 1
        seed_bootstrap_2 += 1

    return results

In [ ]:
# ======================= Algorithm (ISA) =======================

def standardize_rows(X: np.ndarray) -> np.ndarray:
    """Center and scale each row to unit norm.
    
    Input:
    X [np.ndarray]: Numeric matrix to standardize by row.
    
    Output:
    XR [np.ndarray]: Row-standardized matrix with zero-safe normalization."""
    XR = X - X.mean(axis=1, keepdims=True)
    norms = np.linalg.norm(XR, axis=1, keepdims=True)
    norms[norms == 0] = 1  # Avoid division by zero
    XR = XR / norms
    return XR


def standardize_cols(X: np.ndarray) -> np.ndarray:
    """Center and scale each column to unit norm.
    
    Input:
    X [np.ndarray]: Numeric matrix to standardize by column.
    
    Output:
    XC [np.ndarray]: Column-standardized matrix with zero-safe normalization."""
    XC = X - X.mean(axis=0, keepdims=True)
    norms = np.linalg.norm(XC, axis=0, keepdims=True)
    norms[norms == 0] = 1  # Avoid division by zero
    XC = XC / norms
    return XC

In [ ]:
# ============================ Plot for results ===================


def plot_ratio_and_clump_by_state(
    df,
    plot_title: str,
    tau_col="tau",
    state_col="State",
    ratio_col="Proportion",
    clump_col="Number of samples",
    figsize=(12, 7),
    ratio_linewidth=1.8,
    clump_linewidth=1.3,
    ratio_alpha=0.9,
    clump_alpha=0.45,
    marker=None,
    sort_tau=True,
    stop_at_first_nan=True,
    grid=True,
    ratio_reference_y=None,
    show_ratio_reference_line=False,
    tau_reference_values=None,
    show_tau_reference_lines=False,
    reference_line_color="0.3",
    reference_line_width=1.0,
    reference_line_style="--",
    reference_line_alpha=0.5,
    state_colors=None,
    show_clump=True,
    clump_linestyle=":",
    states_legend_anchor=(1.10, 1.0),
    linetype_legend_anchor=(1.10, 0.55),

    # Added font-size controls
    title_fontsize=16,
    axis_label_fontsize=14,
    tick_label_fontsize=12,
    legend_fontsize=12,
    legend_title_fontsize=13,
):
    """Plot overlap ratios and clump sizes by state over tau.
    
    Input:
    df [pd.DataFrame | list]: Results table or list of result tables.
    plot_title [str]: Figure title.
    tau_col [str]: Column name for tau values.
    state_col [str]: Column name for state identifiers.
    ratio_col [str]: Column name for ratio values.
    clump_col [str]: Column name for clump sizes.
    figsize [tuple]: Matplotlib figure size.
    ratio_linewidth [float]: Line width for ratio curves.
    clump_linewidth [float]: Line width for clump-size curves.
    ratio_alpha [float]: Transparency for ratio curves.
    clump_alpha [float]: Transparency for clump-size curves.
    marker [str | None]: Marker style for ratio curves.
    sort_tau [bool]: Whether to sort by state and tau.
    stop_at_first_nan [bool]: Whether to truncate each series at the first NaN.
    grid [bool]: Whether to show the plot grid.
    ratio_reference_y [float | None]: Y-value for the ratio reference line.
    show_ratio_reference_line [bool]: Whether to draw the ratio reference line.
    tau_reference_values [list | None]: Tau values for vertical reference lines.
    show_tau_reference_lines [bool]: Whether to draw tau reference lines.
    reference_line_color [str]: Reference-line color.
    reference_line_width [float]: Reference-line width.
    reference_line_style [str]: Reference-line style.
    reference_line_alpha [float]: Reference-line transparency.
    state_colors [dict | None]: Optional color mapping by state.
    show_clump [bool]: Whether to plot clump sizes on a second axis.
    clump_linestyle [str]: Line style for clump-size curves.
    states_legend_anchor [tuple]: Anchor position for the state legend.
    linetype_legend_anchor [tuple]: Anchor position for the line-type legend.
    title_fontsize [int]: Title font size.
    axis_label_fontsize [int]: Axis-label font size.
    tick_label_fontsize [int]: Tick-label font size.
    legend_fontsize [int]: Legend text font size.
    legend_title_fontsize [int]: Legend title font size.
    
    Output:
    fig [matplotlib.figure.Figure]: Created figure.
    ax1 [matplotlib.axes.Axes]: Primary axis for ratio values.
    ax2 [matplotlib.axes.Axes | None]: Secondary axis for clump sizes."""

    if isinstance(df, (list, tuple)):
        if len(df) == 0:
            raise ValueError("df is an empty list/tuple.")
        df = pd.concat(df, ignore_index=True)
    elif not isinstance(df, pd.DataFrame):
        raise TypeError("df must be a pandas DataFrame or a list of DataFrames.")

    df = df.copy()

    required_cols = {tau_col, state_col, ratio_col, clump_col}
    missing_required = required_cols - set(df.columns)
    if missing_required:
        raise ValueError(f"Missing required columns: {missing_required}")

    if sort_tau:
        df = df.sort_values([state_col, tau_col]).reset_index(drop=True)

    states = sorted(df[state_col].dropna().unique())

    if state_colors is None:
        default_colors = plt.rcParams["axes.prop_cycle"].by_key()["color"]
        state_colors = {
            state: default_colors[i % len(default_colors)]
            for i, state in enumerate(states)
        }

    fig, ax1 = plt.subplots(figsize=figsize)
    ax2 = ax1.twinx() if show_clump else None

    ratio_handles = []

    for state in states:
        sub = (
            df.loc[df[state_col] == state, [tau_col, ratio_col, clump_col]]
            .sort_values(tau_col)
            .reset_index(drop=True)
        )

        sub = sub[sub[tau_col].notna()].reset_index(drop=True)
        if sub.empty:
            continue

        color = state_colors[state]

        # Ratio data
        ratio_sub = sub[[tau_col, ratio_col]].copy()
        if stop_at_first_nan:
            nan_positions = ratio_sub.index[ratio_sub[ratio_col].isna()]
            if len(nan_positions) > 0:
                ratio_sub = ratio_sub.iloc[:nan_positions[0]]
        else:
            ratio_sub = ratio_sub[ratio_sub[ratio_col].notna()]

        if not ratio_sub.empty:
            ax1.plot(
                ratio_sub[tau_col],
                ratio_sub[ratio_col],
                color=color,
                linewidth=ratio_linewidth,
                alpha=ratio_alpha,
                marker=marker,
            )
            ratio_handles.append(
                Line2D(
                    [0], [0],
                    color=color,
                    lw=ratio_linewidth,
                    label=f"State {state}"
                )
            )

        # Clump data
        if show_clump:
            clump_sub = sub[[tau_col, clump_col]].copy()
            if stop_at_first_nan:
                nan_positions = clump_sub.index[clump_sub[clump_col].isna()]
                if len(nan_positions) > 0:
                    clump_sub = clump_sub.iloc[:nan_positions[0]]
            else:
                clump_sub = clump_sub[clump_sub[clump_col].notna()]

            if not clump_sub.empty:
                ax2.plot(
                    clump_sub[tau_col],
                    clump_sub[clump_col],
                    color=color,
                    linewidth=clump_linewidth,
                    alpha=clump_alpha,
                    linestyle=clump_linestyle,
                )

    # Reference lines
    if show_ratio_reference_line and ratio_reference_y is not None:
        ax1.axhline(
            y=ratio_reference_y,
            color=reference_line_color,
            linewidth=reference_line_width,
            linestyle=reference_line_style,
            alpha=reference_line_alpha,
            zorder=0,
        )

    if show_tau_reference_lines and tau_reference_values is not None:
        for tau_value in tau_reference_values:
            ax1.axvline(
                x=tau_value,
                color=reference_line_color,
                linewidth=reference_line_width,
                linestyle=reference_line_style,
                alpha=reference_line_alpha,
                zorder=0,
            )

    # Axis labels and title with larger fonts
    ax1.set_xlabel(r"$\tau$", fontsize=axis_label_fontsize)
    ax1.set_ylabel(ratio_col, fontsize=axis_label_fontsize)
    if show_clump:
        ax2.set_ylabel(clump_col, fontsize=axis_label_fontsize)

    ax1.set_title(plot_title, fontsize=title_fontsize)

    # Tick label sizes
    ax1.tick_params(axis="both", labelsize=tick_label_fontsize)
    if show_clump:
        ax2.tick_params(axis="y", labelsize=tick_label_fontsize)

    if grid:
        ax1.grid(True, alpha=0.3)

    # Leave room on the right for legends
    fig.subplots_adjust(right=0.78)

    # Legend 1: states/colors
    if ratio_handles:
        legend1 = ax1.legend(
            handles=ratio_handles,
            title="States",
            loc="upper left",
            bbox_to_anchor=states_legend_anchor,
            borderaxespad=0.0,
            frameon=True,
            fontsize=legend_fontsize,
            title_fontsize=legend_title_fontsize,
        )
        ax1.add_artist(legend1)

    # Legend 2: line meaning
    if show_clump:
        linetype_handles = [
            Line2D(
                [0], [0],
                color="black",
                lw=ratio_linewidth,
                linestyle="-",
                alpha=0.9,
                label="Proportion of fixed samples"
            ),
            Line2D(
                [0], [0],
                color="black",
                lw=clump_linewidth,
                linestyle=clump_linestyle,
                alpha=clump_alpha,
                label="Number of converged samples"
            ),
        ]
        ax1.legend(
            handles=linetype_handles,
            title="Line type",
            loc="upper left",
            bbox_to_anchor=linetype_legend_anchor,
            borderaxespad=0.0,
            frameon=True,
            fontsize=legend_fontsize,
            title_fontsize=legend_title_fontsize,
        )

    return fig, ax1, ax2

In [ ]:
# ================== Salt and pepper function for augmentation of data =======================

def salt_pepper_batch(imgs, amount=0.02, s_vs_p=0.5, inplace=False, rng=None):
    """Add salt-and-pepper noise to a batch of images.
    
    Input:
    imgs [np.ndarray]: Batch of images to modify.
    amount [float]: Total fraction of pixels to corrupt.
    s_vs_p [float]: Salt-to-pepper noise ratio.
    inplace [bool]: Whether to modify imgs directly.
    rng [np.random.Generator | int | None]: Random generator or seed.
    
    Output:
    x [np.ndarray]: Noisy image batch."""


    x = imgs if inplace else imgs.copy()
    x = x.astype(np.float32, copy=False)

    if rng is None:
        rng = np.random.default_rng()
    elif isinstance(rng, int):
        rng = np.random.default_rng(rng)

    rnd = rng.random(x.shape, dtype=np.float32)

    p_pepper = amount * (1.0 - s_vs_p)
    p_salt = amount * s_vs_p

    pepper = rnd < p_pepper
    salt = rnd > (1.0 - p_salt)

    x[pepper] = 0.0
    x[salt] = 1.0
    return x

# Creat data

This section constructs the base digit datasets before applying augmentation.

Note: The mnist training and test sets must be located in a folde named data

The following cell loads MNIST and filters the selected digit classes.

In [ ]:
mnist_train = datasets.MNIST(root='./data', train=True, download=True, transform=None)
mnist_test = datasets.MNIST(root='./data', train=False, download=True, transform=None)

# Transform to numpy
data_numpy_train = mnist_train.data.numpy()
labels_numpy_train = mnist_train.targets.numpy() 


data_numpy_test = mnist_test.data.numpy()
labels_numpy_test = mnist_test.targets.numpy()

# Extracted integers from the MNIST
a = 3
b = 7

# number of padding in the images
pad = 12

# indexes where 3 and 7 are the targets respectivly
labels_train_a_idx = np.where(labels_numpy_train == a)
labels_train_b_idx = np.where(labels_numpy_train == b)

labels_test_a_idx = np.where(labels_numpy_test == a)
labels_test_b_idx = np.where(labels_numpy_test == b)


# Extract data for the indexes found above
data_train_a = data_numpy_train[labels_train_a_idx]
data_train_b = data_numpy_train[labels_train_b_idx]

data_test_a = data_numpy_test[labels_test_a_idx]
data_test_b = data_numpy_test[labels_test_b_idx]

sigma_list = [0, 90, 180, 270]
theta_list = [0, 120, 240]


data_train_concat, data_train_concat_pad, labels_train_concat_pad = creat_rotated_data_incl_no_pad(data_train_a, data_train_b, sigma_list, theta_list, seed = 1001, pad=pad)#, pad_shift=True, square=False)
data_test_concat, data_test_concat_pad, labels_test_concat_pad = creat_rotated_data_incl_no_pad(data_test_a, data_test_b, sigma_list, theta_list, seed = 2001, pad=pad)#, pad_shift=True)

plot_org_only([0,1,2], data_test_concat)

Continuoing we augment the linear data constructed in the previous cell

The following cell applies reproducible augmentations to the constructed linear data.

In [ ]:
all_train_losses_all_pads = []
all_val_losses_all_pads = []
all_labels_all_pads = []
number_images = 5000
pad = 12


# The datasets which will be augmented
data = data_train_concat_pad
data_Npad = data_train_concat

# Set seed for reproducability (3001)
rng = np.random.default_rng(3001)


# Define functions from albumenation which will be used for augmentation
transform_albo_gauss = A.Compose(
    [
        A.GaussNoise(std_range=(0.05, 0.1), p=1.0),
    ],
    seed=4001,
)

transform_albo_cropout = A.Compose(
    [
        A.CoarseDropout(
            num_holes_range=(3, 7),
            hole_height_range=(5, 10),
            hole_width_range=(5, 5),
            fill=0,
            p=1.0,
        ),
    ],
    seed=5001,
)


# creat arreys to overwrite with the augemnted data
data_reshape = data.reshape(len(data), pad*4+2*28, pad*2+28)
data_Npad_reshape = data_Npad.reshape(len(data_Npad), 2*28, 28)

idx_1 = rng.choice(len(data_reshape), size=number_images, replace=False)
idx_2 = rng.choice(len(data_reshape), size=number_images, replace=False)
idx_3 = rng.choice(len(data_reshape), size=number_images, replace=False)

data_reshape_subset_1 = data_reshape[idx_1]
data_reshape_subset_2 = data_reshape[idx_2]
data_reshape_subset_3 = data_reshape[idx_3]

arr_crop = np.zeros((number_images, pad*4+2*28, pad*2+28))
arr_gaus = np.zeros((number_images, pad*4+2*28, pad*2+28))
arr_sp = np.zeros((number_images, pad*4+2*28, pad*2+28))


# append the different augmentations to different images
for j in range(number_images): 
    str_ = "image"
    
    # ------------ Crop -----------------
    
    albo_crop = transform_albo_cropout(image=data_reshape_subset_1[j].astype(np.float32))
    arr_crop[j] = albo_crop[str_]

    # ------------ Gaussian noize --------------

    albo_gaus = transform_albo_gauss(image=data_reshape_subset_2[j].astype(np.float32))
    albo_gaus_img = albo_gaus[str_]
    albo_gaus_img_norm = albo_gaus_img / np.max(albo_gaus_img)
    arr_gaus[j] = albo_gaus_img_norm

    # ------------ SP noize ----------------
    img_sp = salt_pepper_batch(data_reshape_subset_3[j], amount=0.02, s_vs_p=0.5, inplace=False, rng=rng)

    arr_sp[j] = img_sp

# construct an array of all augmneted dataset and a corresponding target array without augmentation for both with and without padding
data_aug = np.concatenate((arr_crop, arr_gaus, arr_sp), axis=0)
data_aug_targ = np.concatenate((data_reshape[idx_1], data_reshape[idx_2], 
                                data_reshape[idx_3]), axis=0)
data_aug_targ_Npad = np.concatenate((data_Npad_reshape[idx_1], data_Npad_reshape[idx_2], 
                                     data_Npad_reshape[idx_3]), axis=0)


# Shuffel the augmented images
idx_shuffle = rng.choice(len(data_aug), size=len(data_aug), replace=False)

data_aug_shuffle = data_aug[idx_shuffle]
data_aug_targ_shuffle = data_aug_targ[idx_shuffle]
data_aug_targ_Npad_shuffle = data_aug_targ_Npad[idx_shuffle]

# plot one image for the augmented, non augmented with padding and non augmented without padding for the spesifyed index
idx = 1
plot_augmented_comparison(idx, data_aug_shuffle, data_aug_targ_shuffle, data_aug_targ_Npad_shuffle)

# Model

This section defines dataset wrappers, reproducibility utilities, and model training setups.

In [ ]:
class AE_Dataset(Dataset):
    """Wrap NumPy input-target arrays as a PyTorch dataset.
    
    Input:
    X [np.ndarray]: Input sample array.
    y [np.ndarray]: Target sample array.
    
    Output:
    AE_Dataset [Dataset]: Dataset returning tensor input-target pairs."""
    def __init__(self, X, y):
        """Store input and target arrays as float tensors.
        
        Input:
        self [AE_Dataset]: Dataset instance being initialized.
        X [np.ndarray]: Input sample array.
        y [np.ndarray]: Target sample array.
        
        Output:
        None [NoneType]: Stores tensors on the dataset instance."""
        self.X = torch.from_numpy(X).float()
        self.y = torch.from_numpy(y).float()

    def __len__(self):
        """Return the dataset size.
        
        Input:
        self [AE_Dataset]: Dataset instance.
        
        Output:
        length [int]: Number of samples in the dataset."""
        return self.X.shape[0]

    def __getitem__(self, idx):
        """Return one input-target pair.
        
        Input:
        self [AE_Dataset]: Dataset instance.
        idx [int]: Sample index.
        
        Output:
        X_i [torch.Tensor]: Input tensor at idx.
        y_i [torch.Tensor]: Target tensor at idx."""
        return self.X[idx], self.y[idx]
    

def set_seed(seed: int):
    """Set random seeds for reproducible runs.
    
    Input:
    seed [int]: Seed value for random, NumPy, and PyTorch.
    
    Output:
    g [torch.Generator]: Seeded PyTorch generator."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

    g = torch.Generator()
    g.manual_seed(seed)
    return g

## CNN

This section trains the convolutional autoencoder on the padded nonlinear target data.

Hyperparameters from thesis
- batch_size = 64
- weight_decay = 1e-5
- lr = 3e-5
- latent_dim = 64
- epochs = 70

The following cells set training parameters, train the model, and save the trained weights.

In [ ]:
batch_size = 64 
weight_decay = 1e-5
lr = 3e-5
latent_dim = 64
loss_function = nn.MSELoss() 
epochs = 70
pad = 12

seed = 42
generator = set_seed(seed)
rng = np.random.default_rng(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------- Data prep ---------------------

data_train_concat_pad_reshape = data_train_concat_pad.reshape(len(data_train_concat_pad), pad*4+2*28, pad*2+28)
data_train = np.concatenate((data_aug, data_train_concat_pad_reshape), axis=0) 

data_targ = np.concatenate((data_aug_targ, data_train_concat_pad_reshape), axis=0) 

input_shape = (1, data_train.shape[1], data_train.shape[2])


# ---------------- Train/val split ----------------

dataset = AE_Dataset(data_train, data_targ)

train_ratio = 0.8
train_size = int(train_ratio * len(dataset))

indices = np.arange(len(dataset))
rng.shuffle(indices)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

# ---------------- DataLoaders ----------------

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ---------------------Initialize model ----------------------

model_CNN = Conv_AE(input_shape=input_shape, latent_dim=latent_dim).to(device)
optimizer = optim.AdamW(model_CNN.parameters(), lr=lr, weight_decay=weight_decay)

losses_train = []
losses_eval = []

best_loss = np.inf
best_state = None

# ----------------------- Train model -------------------


for epoch in range(epochs):
    loss_train = 0.0
    model_CNN.train()
    
    for x, y in train_loader:
        
        x = x.to(device).unsqueeze(1)
        y = y.to(device).unsqueeze(1)
        
        recon_x, _ = model_CNN(x)
        loss = loss_function(recon_x, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_train += loss.item() * x.size(0)
    
    loss_train /= len(train_loader.dataset)
    
    losses_train.append(loss_train)
    
    model_CNN.eval()
    with torch.no_grad():
        loss_eval = 0.0
        for x, y in val_loader:
            x = x.to(device).unsqueeze(1) 
            y = y.to(device).unsqueeze(1) 

            recon_x, _ = model_CNN(x)
            loss_e = loss_function(recon_x, y)
            
            loss_eval += loss_e.item() * x.size(0)
            
            
        loss_eval /= len(val_loader.dataset)
        losses_eval.append(loss_eval)
        
    
    print(f"Epoch {epoch+1}/{epochs} | train: {loss_train:.6f} | val: {loss_eval:.6f}")


    # -------- checkpoint logic ---------
    if loss_eval < best_loss:      # tiny margin avoids jitter
        best_loss = loss_eval
        best_state  = copy.deepcopy(model_CNN.state_dict())
        best_stat = epoch
    
    
if best_state is not None:    
    model_CNN.load_state_dict(best_state)

# Plot all training losses in one plot
plt.figure(figsize=(10, 5))
plt.plot(losses_train, label='train loss')      # first line
plt.plot(losses_eval,  label='val loss')   
plt.title("Reconstruction loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Best epoch: {best_stat + 1}, best val loss: {best_loss:.6f}")

In [ ]:
# load model

save_dir = "data_inference"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "CNN_ae_be.pt")

torch.save({
    "model_state_dict": model_CNN.state_dict(),
    "input_shape": input_shape,
    "latent_dim": latent_dim,
    "pad": pad,
}, save_path)

print(f"Saved model to: {save_path}")


## CNN diff targ

This section trains the convolutional autoencoder variant where the reconstruction target has a different shape.

Hyperparameters from thesis
- batch_size = 128
- weight_decay = 1e-5
- lr = 1e-3
- latent_dim = 64
- epochs = 100

The following cells configure, train, and save the different-target convolutional autoencoder.

In [ ]:
batch_size = 128
weight_decay = 0.00001
lr = 1e-3
latent_dim = 64
loss_function = nn.MSELoss() 
epochs = 100
pad = 12

seed = 1042
generator = set_seed(seed)
rng = np.random.default_rng(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ------------------- Data prep ---------------------

data_train_concat_pad_reshape = data_train_concat_pad.reshape(len(data_train_concat_pad), pad*4+2*28, pad*2+28)
data_train = np.concatenate((data_aug, data_train_concat_pad_reshape), axis=0)

data_train_concat_reshape = data_train_concat.reshape(len(data_train_concat), 2*28, 28)
data_targ = np.concatenate((data_aug_targ_Npad, data_train_concat_reshape), axis=0)

input_shape = (1, data_train.shape[1], data_train.shape[2])

target_shape = (1, data_targ.shape[1], data_targ.shape[2])


# ---------------- Train/val split ----------------

dataset = AE_Dataset(data_train, data_targ)

train_ratio = 0.8
train_size = int(train_ratio * len(dataset))

indices = np.arange(len(dataset))
rng.shuffle(indices)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

# ---------------- DataLoaders ----------------
generator = torch.Generator()
generator.manual_seed(seed)

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ---------------------Initialize model ----------------------

model_CNN_diff_targ = Conv_AE_diff_targ(input_shape=input_shape, target_shape=target_shape, latent_dim=latent_dim).to(device)
optimizer = optim.AdamW(model_CNN_diff_targ.parameters(), lr=lr, weight_decay=weight_decay)

losses_train = []
losses_eval = []

best_loss = np.inf
best_state = None

# ----------------------- Train model -------------------


for epoch in range(epochs):
    loss_train = 0.0
    model_CNN_diff_targ.train()
    
    for x, y in train_loader:
        
        x = x.to(device).unsqueeze(1)
        y = y.to(device).unsqueeze(1)
        
        recon_x, _ = model_CNN_diff_targ(x)
        loss = loss_function(recon_x, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        loss_train += loss.item() * x.size(0)
    
    loss_train /= len(train_loader.dataset)
    
    losses_train.append(loss_train)
    
    model_CNN_diff_targ.eval()
    with torch.no_grad():
        loss_eval = 0.0
        for x, y in val_loader:
            x = x.to(device).unsqueeze(1) 
            y = y.to(device).unsqueeze(1) 

            recon_x, _ = model_CNN_diff_targ(x)
            loss_e = loss_function(recon_x, y)
            
            loss_eval += loss_e.item() * x.size(0)
            
            
        loss_eval /= len(val_loader.dataset)
        losses_eval.append(loss_eval)
        
    
    print(f"Epoch {epoch+1}/{epochs} | train: {loss_train:.6f} | val: {loss_eval:.6f}")


    # -------- checkpoint logic ---------
    if loss_eval < best_loss:      # tiny margin avoids jitter
        best_loss = loss_eval
        best_state  = copy.deepcopy(model_CNN_diff_targ.state_dict())
        best_stat = epoch
    
    
if best_state is not None:    
    model_CNN_diff_targ.load_state_dict(best_state)

# Plot all training losses in one plot
plt.figure(figsize=(10, 5))
plt.plot(losses_train, label='train loss')      # first line
plt.plot(losses_eval,  label='val loss')   
plt.title("Reconstruction loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

print(f"Best epoch: {best_stat + 1}, best val loss: {best_loss:.6f}")

In [ ]:
# save model
save_dir = "data_inference"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "CNN_diff_targ_ae_be.pt")

torch.save({
    "model_state_dict": model_CNN_diff_targ.state_dict(),
    "input_shape": input_shape,
    "latent_dim": latent_dim,
    "target_shape": target_shape,
    "pad": pad,
}, save_path)

print(f"Saved model to: {save_path}")


## AE simple

This section trains the feedforward autoencoder baseline on vectorized input data.

Hyperparameters from thesis
- batch_size = 64
- weight_decay = 1e-6
- lr = 1e-4
- latent_dim = 64
- epochs = 70

The following cells configure, train, and save the feedforward autoencoder.

In [ ]:
latent_dim = 64
weight_decay = 1e-6
lr = 1e-4
batch_size = 64
epochs = 70
pad = 12

# ----------- Reproducability -------------

seed = 2042
generator = set_seed(seed)
rng = np.random.default_rng(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# -------------- Data prep ------------------

data_aug_reshape = data_aug.reshape(len(data_aug),data_train_concat_pad.shape[1])
data_aug_targ_reshape = data_aug_targ.reshape(len(data_aug_targ), data_train_concat_pad.shape[1])

data_train_ffn = np.concatenate((data_aug_reshape, data_train_concat_pad), axis=0) # data_aug_reshape, 
data_targ_ffn = np.concatenate((data_aug_targ_reshape, data_train_concat_pad), axis=0) #data_aug_reshape, 

input_dim = data_train_ffn.shape[1]

# ---------------- Train/val split ----------------

dataset = AE_Dataset(data_train_ffn, data_targ_ffn)

train_ratio = 0.8
train_size = int(train_ratio * len(dataset))

indices = np.arange(len(dataset))
rng.shuffle(indices)

train_indices = indices[:train_size]
val_indices = indices[train_size:]

train_dataset = Subset(dataset, train_indices)
val_dataset = Subset(dataset, val_indices)

# ---------------- DataLoaders ----------------

train_loader = DataLoader(
    train_dataset,
    batch_size=batch_size,
    shuffle=True,
    generator=generator
)

val_loader = DataLoader(
    val_dataset,
    batch_size=batch_size,
    shuffle=False
)


# ---------------------Initialize model ----------------------

model_FFNN = AutoEncoder_desp(input_dim=input_dim, latent_dim=latent_dim, output_dim=input_dim).to(device)
optimizer = optim.Adam(model_FFNN.parameters(), lr=lr, weight_decay=weight_decay)
loss_function = nn.MSELoss()

losses_train = []
losses_eval = []

best_loss = np.inf

# ----------------------- Train model -------------------


for epoch in range(epochs):
    
    model_FFNN.train()
    train_loss = 0.0
    for x, y in train_loader:
        x = x.to(device)
        y = y.to(device)
        
        recon_y, _ = model_FFNN(x)
        
        loss = loss_function(recon_y, y)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        train_loss += loss.item() * x.size(0)
    
    train_loss /= len(train_loader.dataset)
        
    losses_train.append(train_loss)
    
    model_FFNN.eval()
    with torch.no_grad():
        eval_loss = 0.0
        for x,y in val_loader:
            x = x.to(device)
            y = y.to(device)

            recon_y, _ = model_FFNN(x)
            eval_loss += loss_function(recon_y, y).item() * y.size(0)
            
            
        eval_loss /= len(val_loader.dataset)
        losses_eval.append(eval_loss)
    
    print(f"Epoch {epoch+1}/{epochs} | train: {train_loss:.6f} | val: {eval_loss:.6f}")


    # -------- checkpoint logic ---------
    if eval_loss < best_loss:      # tiny margin avoids jitter
        best_loss = eval_loss
        best_state  = copy.deepcopy(model_FFNN.state_dict())

        best_stat = epoch

# After training is complete, load the best model
model_FFNN.load_state_dict(best_state)

# Plot all training losses in one plot
plt.figure(figsize=(10, 5))
plt.plot(losses_train, label='train loss')      # first line
plt.plot(losses_eval,  label='val loss')   
plt.title("Reconstruction loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

In [ ]:
# save model
save_dir = "data_inference"
os.makedirs(save_dir, exist_ok=True)

save_path = os.path.join(save_dir, "FFNN_ae_be.pt")

torch.save({
    "model_state_dict": model_FFNN.state_dict(),
    "input_dim": input_dim,
    "latent_dim": latent_dim,
    "output_dim": input_dim,
}, save_path)

print(f"Saved model to: {save_path}")


## K-fold cross validation

This section runs cross-validation for the selected model and compares validation performance across folds.

Note under the headling "Data prep" you must choose what model you like to evaluate

CNN: Convoulutional autoencoder with nonlinear targetdata 

FFNN: feedforward autoencoder

CNN_diff_targ: Convoulutional autoencoder with linear target data


Also, hyperparameter must to set to the once presented in the thesis, whisc are:

Convoulutional autoencoder with nonlinear targetdata:
- batch_size = 64
- weight_decay = 1e-5
- lr = 3e-5
- latent_dim = 64
- epochs = 70


feedforward autoencoder:
- batch_size = 64
- weight_decay = 1e-6
- lr = 1e-4
- latent_dim = 64
- epochs = 70


Convoulutional autoencoder with linear target data:
- batch_size = 128
- weight_decay = 1e-5
- lr = 1e-3
- latent_dim = 64
- epochs = 100

All other hypereparameters should not be changed

The following cells prepare the chosen model and evaluate it with k-fold validation.

In [ ]:
# ---------------- Hyperparameters ----------------
batch_size = 128
weight_decay = 1e-5
lr = 1e-3
latent_dim = 64
loss_function = nn.MSELoss()
epochs = 70
pad = 12
n_splits = 5

# ---------------- Reproducibility ----------------
seed = 42
generator = set_seed(seed)   
rng = np.random.default_rng(seed)

torch.manual_seed(seed)
torch.cuda.manual_seed_all(seed)
random.seed(seed)
np.random.seed(seed)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# ---------------- Data prep ----------------

# model_type has to be choosen acordingly to what model you have loaded
# You can choos between the following: 
# "CNN" ~ Convolutional autoencoder witn nonlinear target
# "FFNN" ~ feedforward autoencoder with nonlinear target
# "CNN_diff_targ" ~ Convolutionsl autoencoder with linear target

model_type = "CNN" 


if model_type == "CNN":
    data_train_concat_pad_reshape = data_train_concat_pad.reshape(len(data_train_concat_pad), pad*4+2*28, pad*2+28)
    data_train = np.concatenate((data_aug, data_train_concat_pad_reshape), axis=0) 

    data_targ = np.concatenate((data_aug_targ, data_train_concat_pad_reshape), axis=0) 
    
    input_shape = (1, data_train.shape[1], data_train.shape[2])
    target_shape = (1, data_targ.shape[1], data_targ.shape[2])
    
elif model_type == "CNN_diff_targ":
    data_train_concat_pad_reshape = data_train_concat_pad.reshape(len(data_train_concat_pad), pad*4+2*28, pad*2+28)
    data_train = np.concatenate((data_aug, data_train_concat_pad_reshape), axis=0)

    data_train_concat_reshape = data_train_concat.reshape(len(data_train_concat), 2*28, 28)
    data_targ = np.concatenate((data_aug_targ_Npad, data_train_concat_reshape), axis=0)
    
    input_shape = (1, data_train.shape[1], data_train.shape[2])
    target_shape = (1, data_targ.shape[1], data_targ.shape[2])
    
elif model_type == "FFNN":
    data_aug_reshape = data_aug.reshape(len(data_aug),data_train_concat_pad.shape[1])
    data_aug_targ_reshape = data_aug_targ.reshape(len(data_aug_targ), data_train_concat_pad.shape[1])

    data_train = np.concatenate((data_aug_reshape, data_train_concat_pad), axis=0) # data_aug_reshape, 
    data_targ = np.concatenate((data_aug_targ_reshape, data_train_concat_pad), axis=0) #data_aug_reshape, 
    
    input_shape = data_train.shape[1]
    target_shape = data_targ.shape[1]




dataset = AE_Dataset(data_train, data_targ)

# ---------------- K-fold CV ----------------
kf = KFold(n_splits=n_splits, shuffle=True, random_state=seed)

fold_results = []
all_fold_train_losses = []
all_fold_val_losses = []

for fold, (train_idx, val_idx) in enumerate(kf.split(np.arange(len(dataset))), 1):
    print(f"\n{'='*20} Fold {fold}/{n_splits} {'='*20}")

    train_dataset = Subset(dataset, train_idx)
    val_dataset = Subset(dataset, val_idx)

    fold_generator = torch.Generator()
    fold_generator.manual_seed(seed + fold)

    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        generator=fold_generator
    )

    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False
    )
    
    
    if model_type == "CNN":
        model =  Conv_AE(
            input_shape=input_shape, 
            latent_dim=latent_dim
        ).to(device)
    
    elif model_type == "CNN_diff_targ":
        model = Conv_AE_diff_targ(
            input_shape=input_shape,
            target_shape=target_shape,
            latent_dim=latent_dim
        ).to(device)

        
    elif model_type == "FFNN":
        model = AutoEncoder_desp(
            input_dim=input_shape, 
            latent_dim=latent_dim, 
            output_dim=input_shape
        ).to(device)



    optimizer = optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay
    )

    losses_train = []
    losses_val = []

    best_loss = np.inf
    best_state = None
    best_epoch = -1

    for epoch in range(epochs):
        # -------- train --------
        model.train()
        loss_train = 0.0

        for x, y in train_loader:
            x = x.to(device).unsqueeze(1)
            y = y.to(device).unsqueeze(1)

            recon_x, _ = model(x)
            loss = loss_function(recon_x, y)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()

            loss_train += loss.item() * x.size(0)

        loss_train /= len(train_loader.dataset)
        losses_train.append(loss_train)

        # -------- validate --------
        model.eval()
        loss_val = 0.0

        with torch.no_grad():
            for x, y in val_loader:
                x = x.to(device).unsqueeze(1)
                y = y.to(device).unsqueeze(1)

                recon_x, _ = model(x)
                loss_e = loss_function(recon_x, y)

                loss_val += loss_e.item() * x.size(0)

        loss_val /= len(val_loader.dataset)
        losses_val.append(loss_val)

        print(f"Epoch {epoch+1}/{epochs} | train: {loss_train:.6f} | val: {loss_val:.6f}")

        # -------- save best fold model --------
        if loss_val < best_loss:
            best_loss = loss_val
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1


    fold_results.append({
        "fold": fold,
        "best_val_loss": best_loss,
        "best_epoch": best_epoch
    })

    all_fold_train_losses.append(losses_train)
    all_fold_val_losses.append(losses_val)

# ---------------- Summary ----------------
best_val_losses = [r["best_val_loss"] for r in fold_results]

print("\nK-fold results:")
for r in fold_results:
    print(f"Fold {r['fold']}: best val loss = {r['best_val_loss']:.6f} at epoch {r['best_epoch']}")

print(f"\nMean best val loss: {np.mean(best_val_losses):.6f}")
print(f"Std best val loss:  {np.std(best_val_losses):.6f}")

# ---------------- Plot validation curves ----------------
plt.figure(figsize=(10, 5))
for i in range(n_splits):
    plt.plot(all_fold_val_losses[i], label=f'fold {i+1} val', alpha=0.8)

plt.title("Validation loss per fold")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.legend(fontsize=8)
plt.grid(True)
plt.tight_layout()
plt.show()

# Test

This section loads trained models, evaluates reconstruction quality, and prepares latent representations for benchmark and ISA tests.



<!-- ### Load and run model -->


### Load and run model

This section loads a selected trained model and computes latent representations and test-set reconstruction error.

1. Load the model you want to construct a latent representation for, 
2. Then you have the option to compute the latent representation of the test set, and/or compute the generalization error of the test set. 

Use the following cells to choose the saved model and produce the outputs needed by later tests.

In [ ]:
# ============= Load FFNN model ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
checkpoint = torch.load("models_for_inference/FFNN_ae_best.pt", map_location=device, weights_only=True)

model = AutoEncoder_desp(
    input_dim=checkpoint["input_dim"],
    latent_dim=checkpoint["latent_dim"],
    output_dim=checkpoint["output_dim"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
# ============= Load CNN model ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
checkpoint = torch.load("models_for_inference/CNN_ae_best.pt", map_location=device, weights_only=True)

model = Conv_AE(
    input_shape=checkpoint["input_shape"],
    latent_dim=checkpoint["latent_dim"]
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
# ============= Load CNN diff targ model ==================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
checkpoint = torch.load("models_for_inference/CNN_diff_targ_ae_best.pt", map_location=device, weights_only=True)

model = Conv_AE_diff_targ(
    input_shape=checkpoint["input_shape"],
    latent_dim=checkpoint["latent_dim"],
    target_shape = checkpoint["target_shape"],
).to(device)

model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

In [ ]:
# =================== Compute latent data ==================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

pad = 12


# model_type has to be choosen acordingly to what model you have loaded
# You can choos between the following: 
# "CNN" ~ Convolutional autoencoder witn nonlinear target
# "FFNN" ~ feedforward autoencoder with nonlinear target
# "CNN_diff_targ" ~ Convolutionsl autoencoder with linear target
model_type = "CNN_diff_targ" 


if model_type == "CNN":
    data_test = data_test_concat_pad.reshape(data_test_concat_pad.shape[0], 56 + pad*4, 28 + pad*2)
    data_test_targ = data_test_concat_pad.reshape(data_test_concat_pad.shape[0], 56 + pad*4, 28 + pad*2)
    
elif model_type == "CNN_diff_targ":
    data_test = data_test_concat_pad.reshape(data_test_concat_pad.shape[0], 56 + pad*4, 28 + pad*2)
    data_test_targ = data_test_concat.reshape(data_test_concat.shape[0], 56, 28)
    
elif model_type == "FFNN":
    data_test = data_test_concat_pad
    data_test_targ = data_test_concat_pad
    
else:
    print("Not valid data type")

batch_size = len(data_test)

dataset = AE_Dataset(data_test, data_test_targ)


indices = np.arange(len(dataset))
dataset = Subset(dataset, indices)

loader = DataLoader(
    dataset,
    batch_size=batch_size,
    shuffle=False
)

model = model.to(device)
model.eval()
with torch.no_grad():
    for x, y in loader:
        x = x.to(device).unsqueeze(1) 
        y = y.to(device).unsqueeze(1) 

        recon_x, latent = model(x)
        
        data_mod = x
        label_mod = y
        
# latern representation   
latent_data = latent

In [ ]:
# ================== Computetest generalization error =====================

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

loss_function = nn.MSELoss()
pad = 12

# model_type has to be choosen acordingly to what model you have loaded
# You can choos between the following: 
# "CNN" ~ Convolutional autoencoder witn nonlinear target
# "FFNN" ~ feedforward autoencoder with nonlinear target
# "CNN_diff_targ" ~ Convolutionsl autoencoder with linear target
model_type = "CNN_diff_targ" 


if model_type == "CNN":
    data_test = data_test_concat_pad.reshape(data_test_concat_pad.shape[0], 56 + pad*4, 28 + pad*2)
    data_test_targ = data_test_concat_pad.reshape(data_test_concat_pad.shape[0], 56 + pad*4, 28 + pad*2)
    
elif model_type == "CNN_diff_targ":
    data_test = data_test_concat_pad.reshape(data_test_concat_pad.shape[0], 56 + pad*4, 28 + pad*2)
    data_test_targ = data_test_concat.reshape(data_test_concat.shape[0], 56, 28)
    
elif model_type == "FFNN":
    data_test = data_test_concat_pad
    data_test_targ = data_test_concat_pad
    
else:
    print("Not valid data type")


batch_size_test = 64

dataset = AE_Dataset(data_test, data_test_targ)

losses_eval = []
indices = np.arange(len(dataset))
test_dataset = Subset(dataset, indices)

test_loader = DataLoader(
    test_dataset,
    batch_size=batch_size_test,
    shuffle=False
)

losses_val = []

model.eval()
val_loss = 0.0

with torch.no_grad():
    for x, y in test_loader:
        x = x.to(device).unsqueeze(1) 
        y = y.to(device).unsqueeze(1) 

        recon_y, _ = model(x)
        val_loss += loss_function(recon_y, y).item() * y.size(0)

val_loss /= len(test_loader.dataset)
losses_val.append(val_loss)


generalization_error = val_loss

In [ ]:
# ============= Save latent data ==================

np.save("data_for_inference/data_latent_CNN_lin_targ.npy", latent.cpu().numpy())

In [ ]:
# ============ Plot the input data and corresponding reconstruction ==================

idx = 110
plot_org_only([idx], recon_x.squeeze(1).cpu())
plot_org_only([idx], data_test_concat_pad)

### Benchmark test

This section computes baseline cluster statistics from state initializations and model latent representations.

1. compute the state initialization list from the label array
2. Rename variable name for n¨linear and nonlinear dataset
3. Set parameters for the benchmark test
4. Compute banckmark test on the desired dataset. Use latent representation from cells under "Load and run model"

Use the following cells to build benchmark state lists and compare the selected representation.

In [ ]:
# ========== compute state initialization list =============

labels = labels_test_concat_pad


# Compute the array containing the the state initialization lists
state_initializations = []

state_initializations.append( np.where( np.isin( labels, np.array([0,1,2,3]))))
state_initializations.append( np.where( np.isin( labels, np.array([4,5,6,7]))))
state_initializations.append( np.where( np.isin( labels, np.array([8,9,10,11]))))
state_initializations.append( np.where( np.isin( labels, np.array([0,4,8]))))
state_initializations.append( np.where( np.isin( labels, np.array([1,5,9]))))
state_initializations.append( np.where( np.isin( labels, np.array([2,6,10]))))
state_initializations.append( np.where( np.isin( labels, np.array([3,7,11]))))

In [ ]:
# creat new variable name for teh linear of nonlinear dataset used in the experiment.
# For latent data, use the latent representation obtained under "Load and run model".

data_linear = data_test_concat
data_nonlinear = data_test_concat_pad


In [ ]:
# Prameters used in the computation of the benchmark test

alpha = 0.05
num_itr = 100
n_bootstrap = 10000
number_init = 1000
range_init = 400

In [ ]:
# ====================================== compute benchmark test =============================

# list of thresholds
T = np.arange(0.0, 2.6, 0.1)

# data to compute benchmark test on
data = None

# NOTE: seeds for reproducability are set in the function "run_ratio_bootstrap_analysis"



Results = run_ratio_bootstrap_analysis(data = data, ## !change str! all_results_vol_2['data_linear']
                                        T = T, ##
                                        number_init = number_init,
                                        num_itr = num_itr,
                                        alpha = alpha,
                                        n_bootstrap = n_bootstrap,
                                        range_init = range_init)

list_pandas = []


for i in range(len(Results)):
    
    # convert from dictonary to pandas dataframe
    list_pandas.append({
        'tau': T[i],
        "Proportion_mean": Results[i]['ratio_1_BS_stat']['sample_mean'],
        "Proportion_BS_std": Results[i]['ratio_1_BS_stat']['bootstrap_std'],
        "Number of samples mean": np.floor(Results[i]["converges_set_size_stat"]['sample_mean']),
        "Number of samples BS std": np.floor(Results[i]['converges_set_size_stat']['bootstrap_std']),
    })

    
    print("\ntau: ", T[i])
    print("Proportion mean:", Results[i]['ratio_1_BS_stat']['sample_mean'], "Bootstrap std: ", Results[i]['ratio_1_BS_stat']['bootstrap_std'])
    
    # Include the converged sample size if deired
    # print("Converged sample set size:", Results[i]["converges_set_size_stat"]['sample_mean'], "Bootstrap std: ", Results[i]['converges_set_size_stat']['bootstrap_std'])

df_dict_ratio = pd.DataFrame(list_pandas)

### ISA-test

This section runs the ISA procedure, compares convergence behavior, and plots the resulting ratios and sample counts.


1. Compute the state initialization list from the label array
2. Compute the sample lists from the state initializations in the following cell
3. Rename variable name for n¨linear and nonlinear dataset
4. Compute the ISA test
5. Plot teh ISA test

Use the following cells to configure ISA, run the iterations, and plot the final results.

In [ ]:
# ========== compute state initialization list =============

labels = labels_test_concat_pad


# Compute the array containing the the state initialization lists
state_initializations = []

state_initializations.append( np.where( np.isin( labels, np.array([0,1,2,3]))))
state_initializations.append( np.where( np.isin( labels, np.array([4,5,6,7]))))
state_initializations.append( np.where( np.isin( labels, np.array([8,9,10,11]))))
state_initializations.append( np.where( np.isin( labels, np.array([0,4,8]))))
state_initializations.append( np.where( np.isin( labels, np.array([1,5,9]))))
state_initializations.append( np.where( np.isin( labels, np.array([2,6,10]))))
state_initializations.append( np.where( np.isin( labels, np.array([3,7,11]))))

In [ ]:
# ====================== Compute the sample set of the state initializations =============

rng = np.random.default_rng(43)
size = 400

short_state_initializations = []
for init in state_initializations:
    temp = rng.choice(init[0], size=size, replace=False)
    short_state_initializations.append((temp,))

In [ ]:
# creat new variable name for teh linear of nonlinear dataset used in the experiment.
# For latent data, use the latent representation obtained under "Load and run model".

data_linear = data_test_concat
data_nonlinear = data_test_concat_pad

In [ ]:
# =================== Compute ISA test =================

# Chose dataset to compute the ISA-test on
data = None

list_pandas = []

# Choose intervalu of thresholds tau
tau_list = np.arange(0.0, 3.2, 0.1)


for tau in tau_list:
    converged_act_list, _ = find_act_given_initialization(data, 
                                                        short_state_initializations, 
                                                        tau,
                                                        num_itr=100)

    dict_init, dict_conv_length = ratio_inference(state_initializations, converged_act_list)


    # convert from dictonary to pandas dataframe
    df_dict_ratio = pd.DataFrame({
        'tau': tau,
        'State': range(len(dict_init["Ratio convCAPinit-act to conv-act"])),
        "Proportion": dict_init["Ratio convCAPinit-act to conv-act"],
        "Number of samples": dict_conv_length["Converged sample length"]
    })
    
    
    list_pandas.append(df_dict_ratio)

list_pandas

In [ ]:
# ================= Plot the results from the ISA-test ==============

my_state_colors = {
    0: "#bac3d6",
    1: "#5CBCC5",
    2: "#046A92",
    3: "#e0e00fed",
    4: "#96e212d6",
    5: "#519C06",
    6: "#287006",
}


fig, ax1, ax2 = plot_ratio_and_clump_by_state(
    df=list_pandas,
    plot_title=r"Evaluation of ISA for different values of $\tau$ with partial state initializations",
    state_colors=my_state_colors,
    tau_reference_values=[1.0,2.5],
    show_tau_reference_lines=True,
    ratio_reference_y=[0.025],
    show_ratio_reference_line = True,
    stop_at_first_nan = False,
    
    title_fontsize=14,
    axis_label_fontsize=14,
    tick_label_fontsize=14,
    legend_fontsize=14,
    legend_title_fontsize=14,
    
    states_legend_anchor=(1.15, 0.8),
    linetype_legend_anchor=(1.10, 1.0),
    #clump_col="Converged set size",
    #ratio_col="Fraction of fixed initializations",
)
plt.show()